# LeetCode 740 — Delete and Earn bằng Python

Link đề: https://leetcode.com/problems/delete-and-earn/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Dynamic Programming

Skeleton LeetCode:

```python
class Solution:
    def deleteAndEarn(self, nums: List[int]) -> int:
        ...
```

Ghi chú: bài này là **House Robber** (LeetCode 198) "đội lốt". Phần khó nhất không phải quy hoạch động mà là **nhận ra phép quy dẫn**: gom các số bằng nhau thành một "nhà" có giá trị `v * (số lần xuất hiện)`, và hai giá trị **liền kề** (`v` và `v + 1`) không được lấy cùng lúc. Notebook đi từ vét cạn quay lui, qua đệ quy có nhớ, quy hoạch động theo mảng giá trị, rồi quy hoạch động chạy trên các giá trị phân biệt đã sắp xếp với bộ nhớ `O(1)` thêm.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mỗi lần chọn một phần tử `nums[i]` ta xóa nó, nhận `nums[i]` điểm, rồi **bắt buộc xóa mọi** phần tử bằng `nums[i] - 1` và `nums[i] + 1`; cần tối đa tổng điểm.
2. Nắm các công cụ Python: `collections.Counter`, `sorted(set(...))`, mảng "đếm theo giá trị" (bucket array), `functools.lru_cache`, `max`, gán nhiều biến cùng lúc `a, b = b, a + 1`.
3. Đi từ **quay lui chọn/bỏ từng giá trị** → **đệ quy có nhớ trên trục giá trị** → **DP mảng `dp[x]`** → **DP cuộn trên các giá trị phân biệt (có xử lý khoảng trống)**.
4. Hiểu mấu chốt: khi đã lấy một giá trị `v` thì nên lấy **tất cả** các bản sao của `v` (vì chúng chỉ làm điểm tăng và xóa cùng một tập hàng xóm); vậy bài toán chỉ còn "chọn tập giá trị không có hai giá trị liền kề, tối đa tổng `v * cnt[v]`" — đúng bài **House Robber**.
5. Tránh các lỗi kinh điển: chạy House Robber trên mảng `nums` thô, quên nhân với số lần xuất hiện, coi hai giá trị cách nhau `>= 2` là xung đột, và dùng tham lam "lấy giá trị nhiều điểm nhất trước".

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Bạn có thể thực hiện thao tác sau **bao nhiêu lần tùy ý**:

- Chọn một phần tử `nums[i]` và xóa nó, nhận `nums[i]` điểm. Sau đó **phải xóa mọi phần tử bằng `nums[i] - 1` và mọi phần tử bằng `nums[i] + 1`**.

Trả về **số điểm tối đa** có thể nhận.

**Ví dụ 1**

- Input: `nums = [3,4,2]`
- Output: `6`
- Giải thích: xóa `4` được `4` điểm, kéo theo xóa `3`; còn `[2]`, xóa `2` được thêm `2` điểm. Tổng `6`.

**Ví dụ 2**

- Input: `nums = [2,2,3,3,3,4]`
- Output: `9`
- Giải thích: xóa một số `3` được `3` điểm, kéo theo xóa mọi số `2` và `4`; còn `[3,3]`. Xóa tiếp hai số `3` nữa, mỗi lần `3` điểm. Tổng `9`.

**Ràng buộc:**

- `1 <= nums.length <= 2 * 10^4`
- `1 <= nums[i] <= 10^4`

Chú ý: sau khi xóa một số `v` và hàng xóm của nó, các bản sao còn lại của `v` **vẫn còn** (như ví dụ 2: còn `[3,3]`), nên ta có thể tiếp tục xóa chúng mà không bị cấm.

In [1]:
from typing import List
from collections import Counter
from functools import lru_cache
import random
import sys
import time

sys.setrecursionlimit(50000)       # cần cho các cách đệ quy sâu tới 10^4 giá trị (Cách 2)

examples = [
    ([3, 4, 2], 6),
    ([2, 2, 3, 3, 3, 4], 9),
]
for nums, expected in examples:
    print(nums, "->", expected, "| số phần tử:", len(nums), "| Counter:", dict(sorted(Counter(nums).items())))

[3, 4, 2] -> 6 | số phần tử: 3 | Counter: {2: 1, 3: 1, 4: 1}
[2, 2, 3, 3, 3, 4] -> 9 | số phần tử: 6 | Counter: {2: 2, 3: 3, 4: 1}


## 3. Kiến thức nền

### 3.1. `Counter` và `sorted(set(...))`

`Counter(nums)` đếm số lần xuất hiện của từng giá trị (một loại từ điển). `sorted(set(nums))` hoặc `sorted(counter)` cho các giá trị **phân biệt** theo thứ tự tăng dần. Với bài này `Counter` cho ta "số bản sao" của mỗi giá trị.

In [2]:
nums = [2, 2, 3, 3, 3, 4]
cnt = Counter(nums)
print(cnt, "| cnt[3] =", cnt[3], "| cnt[9] =", cnt[9])     # khóa chưa có -> 0, không lỗi
print(sorted(cnt), "| các cặp:", sorted(cnt.items()))

Counter({3: 3, 2: 2, 4: 1}) | cnt[3] = 3 | cnt[9] = 0
[2, 3, 4] | các cặp: [(2, 2), (3, 3), (4, 1)]


### 3.2. Mảng "đếm theo giá trị" (bucket array)

Vì `1 <= nums[i] <= 10^4`, ta có thể dùng một **mảng** `pts` có chỉ số là giá trị: `pts[v]` = tổng điểm nếu lấy mọi bản sao của `v` = `v * (số lần v xuất hiện)`. Mảng cỡ `max(nums) + 1`. Cách này thay từ điển bằng chỉ số, nhanh và gọn, và còn tiện cho việc xét "hàng xóm" `v - 1`, `v + 1` (chỉ số liền nhau).

In [3]:
def build_points(nums):
    pts = [0] * (max(nums) + 1)
    for x in nums:
        pts[x] += x                # mỗi bản sao của x góp thêm x điểm
    return pts

print(build_points([2, 2, 3, 3, 3, 4]))     # [0, 0, 4, 9, 4]
print(build_points([3, 4, 2]))              # [0, 0, 2, 3, 4]

[0, 0, 4, 9, 4]
[0, 0, 2, 3, 4]


### 3.3. Bài "House Robber": chọn không liền kề để tối đa tổng

**House Robber** (LeetCode 198): có dãy các căn nhà, mỗi nhà có `a[i]` tiền, không được cướp hai nhà **liền nhau**. Tối đa tổng tiền?

Gọi `f[i]` là tiền tối đa khi chỉ xét `i` nhà đầu. Với nhà thứ `i`: hoặc **bỏ qua** (`f[i - 1]`), hoặc **lấy** nó và nhà `i - 1` bị cấm (`f[i - 2] + a[i]`). Vậy `f[i] = max(f[i - 1], f[i - 2] + a[i])`.

In [4]:
def house_robber(a):
    prev2, prev1 = 0, 0                     # f[i-2], f[i-1]
    for x in a:
        prev2, prev1 = prev1, max(prev1, prev2 + x)
    return prev1

print(house_robber([2, 7, 9, 3, 1]))        # 12 = 2 + 9 + 1
print(house_robber([1, 2, 3, 1]))           # 4  = 1 + 3

12
4


### 3.4. `lru_cache` và đệ quy có nhớ

`@lru_cache(None)` đặt trên một hàm đệ quy sẽ **nhớ kết quả** theo tham số, nên mỗi tham số chỉ tính một lần. Lưu ý đệ quy sâu (hàng nghìn tầng) cần nâng `sys.setrecursionlimit`, và với độ sâu cỡ `10^4` thì `lru_cache` (cài bằng C) có thể lỗi dù đã nâng giới hạn; khi đó tự cài bộ nhớ đệm bằng `dict`.

In [5]:
@lru_cache(None)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print(fib(80))
print(fib.cache_info().misses, "lần tính thật (thay vì hàng chục tỉ nếu không nhớ)")

23416728348467685
81 lần tính thật (thay vì hàng chục tỉ nếu không nhớ)


### 3.5. Gán nhiều biến cùng lúc

`a, b = b, a + 1` tính **vế phải trước** rồi mới gán, nên không cần biến tạm. Ta dùng nó để "cuộn" hai biến trạng thái của DP mà không cần mảng.

In [6]:
a, b = 1, 2
a, b = b, a + b
print(a, b)         # 2 3  (vế phải tính từ a, b cũ)

2 3


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Lấy **ví dụ 2** `[2,2,3,3,3,4]`. Đếm: `2` xuất hiện `2` lần, `3` xuất hiện `3` lần, `4` xuất hiện `1` lần.

- Lấy hết các số `2` được `2 * 2 = 4` điểm, nhưng buộc xóa các `3` (và `1`, không có).
- Lấy hết các số `3` được `3 * 3 = 9` điểm, buộc xóa các `2` và `4`.
- Lấy hết các số `4` được `4` điểm, buộc xóa các `3`.

Có các lựa chọn không xung đột: `{2, 4}` cho `4 + 4 = 8`; `{3}` cho `9`. Tốt nhất là `9`. Ta thấy:

1. Đã lấy `v` thì nên lấy **hết** các bản sao của `v`: các bản sao khác không bị xóa bởi việc lấy `v`, và việc xóa chúng không xóa thêm gì ngoài `v - 1`, `v + 1` (đã bị xóa rồi). Nên không có lý do bỏ lại.
2. Hai giá trị **liền kề** (`v` và `v + 1`) **không thể cùng được lấy** (lấy một cái thì cái kia bị xóa sạch). Hai giá trị cách nhau `>= 2` thì **độc lập**.
3. Vậy bài toán trở thành: chọn tập giá trị **không có hai giá trị liền kề** để tối đa `Σ v * cnt[v]` — **House Robber** trên trục giá trị.

Ví dụ 1 `[3,4,2]`: điểm `{2: 2, 3: 3, 4: 4}`; chọn `{2, 4}` được `6` (không liền kề), hơn `{3}` được `3`.

In [7]:
nums = [2, 2, 3, 3, 3, 4]
pts = build_points(nums)
print("điểm theo giá trị:", {v: p for v, p in enumerate(pts) if p})
print("House Robber trên pts (chỉ số = giá trị):", house_robber(pts))
print("ví dụ 1:", house_robber(build_points([3, 4, 2])))

điểm theo giá trị: {2: 4, 3: 9, 4: 4}
House Robber trên pts (chỉ số = giá trị): 9
ví dụ 1: 6


## 5. Cách 1 — Brute Force: quay lui chọn/bỏ từng giá trị

### Ý tưởng

Sắp xếp các giá trị **phân biệt** `vals`. Xét lần lượt từng giá trị, có hai lựa chọn:

- **Bỏ** `vals[k]`: chuyển sang `vals[k + 1]`.
- **Lấy** `vals[k]` (nhận `vals[k] * cnt` điểm): bắt buộc bỏ giá trị `vals[k] + 1` nếu có mặt, nên nhảy tới giá trị kế tiếp sau đó.

Lấy tối đa của hai lựa chọn. Không nhớ gì cả nên số nhánh tăng theo cấp số nhân.

### Code

In [8]:
def earn_backtrack(nums):
    cnt = Counter(nums)
    vals = sorted(cnt)
    calls = [0]

    def go(k):
        calls[0] += 1
        if k >= len(vals):
            return 0
        best = go(k + 1)                                   # bỏ vals[k]
        nk = k + 1
        if nk < len(vals) and vals[nk] == vals[k] + 1:     # lấy vals[k] thì vals[k] + 1 bị xóa
            nk += 1
        best = max(best, vals[k] * cnt[vals[k]] + go(nk))  # lấy hết các bản sao của vals[k]
        return best

    ans = go(0)
    return ans, calls[0]

for nums, expected in examples:
    print(nums, "->", earn_backtrack(nums)[0], "(kỳ vọng", expected, ")")

[3, 4, 2] -> 6 (kỳ vọng 6 )
[2, 2, 3, 3, 3, 4] -> 9 (kỳ vọng 9 )


### Độ phức tạp và nút thắt

- Time: với `k` giá trị phân biệt **liên tiếp** (`1, 2, ..., k`), số lời gọi đệ quy tăng như dãy Fibonacci: khoảng `φ^k` với `φ ≈ 1.618`. Rất nhanh vượt tầm với `k = 40`.
- Space: `O(k)` ngăn xếp đệ quy.

**Nút thắt:** cùng một trạng thái "bắt đầu xét từ `vals[k]`" bị tính đi tính lại rất nhiều lần (từ nhiều đường đi khác nhau). Xem số lời gọi.

In [9]:
for k in [5, 10, 15, 20, 25]:
    nums = list(range(1, k + 1))
    t0 = time.perf_counter()
    ans, calls = earn_backtrack(nums)
    print(f"k={k}: đáp án {ans}, số lời gọi {calls}, thời gian {time.perf_counter() - t0:.4f}s")

k=5: đáp án 9, số lời gọi 25, thời gian 0.0000s
k=10: đáp án 30, số lời gọi 287, thời gian 0.0000s
k=15: đáp án 64, số lời gọi 3193, thời gian 0.0005s
k=20: đáp án 110, số lời gọi 35421, thời gian 0.0069s
k=25: đáp án 169, số lời gọi 392835, thời gian 0.0570s


Số lời gọi tăng khoảng `1.6` lần mỗi khi `k` tăng thêm `1`. Với `k = 10^4` giá trị phân biệt thì hoàn toàn không thể.

## 6. Ý tưởng thuật toán / bước đột phá

Đúng như quan sát ở mục 5: kết quả của `go(k)` chỉ phụ thuộc `k`, và có tối đa `k_tổng` trạng thái khác nhau. **Nhớ lại kết quả** (memoization) hoặc tính **theo thứ tự tăng dần** (quy hoạch động) sẽ cho `O(k)`.

Cụ thể, đặt trên **trục giá trị** (không chỉ trên các giá trị có mặt): `pts[x]` như mục 3.2 và

> `dp[x]` = điểm tối đa khi chỉ xét các giá trị `<= x`.
>
> `dp[x] = max(dp[x - 1], dp[x - 2] + pts[x])`.

Ý nghĩa: hoặc **không lấy** giá trị `x` (`dp[x - 1]`), hoặc **lấy** `x` thì phải bỏ `x - 1` nên cộng vào `dp[x - 2]`. Giá trị không có mặt có `pts[x] = 0`, nên công thức vẫn đúng (khoảng trống tự động cho phép chọn cả hai bên). Cơ sở: `dp[0] = 0`, `dp[1] = pts[1]`. Đáp án `dp[max(nums)]`.

Hai chi phí: duyệt `nums` một lần để tạo `pts` (`O(n)`), rồi quét trục giá trị (`O(max(nums))`).

In [10]:
nums = [2, 2, 3, 3, 3, 4]
pts = build_points(nums)
dp = [0] * len(pts)
dp[1] = pts[1]
for x in range(2, len(pts)):
    dp[x] = max(dp[x - 1], dp[x - 2] + pts[x])
    print(f"x={x}: pts={pts[x]}  dp[{x}] = max(dp[{x - 1}]={dp[x - 1]}, dp[{x - 2}]={dp[x - 2]} + {pts[x]}) = {dp[x]}")
print("đáp án:", dp[-1])

x=2: pts=4  dp[2] = max(dp[1]=0, dp[0]=0 + 4) = 4
x=3: pts=9  dp[3] = max(dp[2]=4, dp[1]=0 + 9) = 9
x=4: pts=4  dp[4] = max(dp[3]=9, dp[2]=4 + 4) = 9
đáp án: 9


## 7. Các cách tiếp theo

Ta có 4 cách khác nhau thực sự:

| Cách | Ý tưởng |
|---|---|
| 1 | Quay lui chọn/bỏ từng giá trị phân biệt (mục 5) |
| 2 | **Đệ quy có nhớ** trên trục giá trị: `f(x) = max(f(x-1), f(x-2) + pts[x])` |
| 3 | **DP mảng** `dp[x]` quét từ `1` tới `max(nums)` (cách nộp) |
| 4 | **DP cuộn** trên các giá trị **phân biệt đã sắp xếp**, xử lý khoảng trống bằng so sánh `v == prev + 1` |

### Cách 2 — Đệ quy có nhớ

**Giải thích từng đoạn:**

- `pts` như mục 3.2; `f(x)` = điểm tối đa dùng các giá trị `<= x`.
- Cơ sở: `f(x) = 0` khi `x <= 0`; `f(1) = pts[1]`.
- Chuyển trạng thái: `f(x) = max(f(x - 1), f(x - 2) + pts[x])`.
- Từ điển `memo` giúp mỗi `x` chỉ tính một lần (ta tự cài thay vì dùng `lru_cache`, vì `lru_cache` bọc hàm bằng mã C và đệ quy sâu `10^4` tầng có thể vượt giới hạn ngăn xếp C của Python). Độ sâu đệ quy tới `max(nums) <= 10^4` nên phải nâng giới hạn đệ quy bằng `sys.setrecursionlimit`.

### Code demo

In [11]:
def earn_memo(nums):
    pts = build_points(nums)
    memo = {}                                  # tự cài bộ nhớ đệm bằng từ điển (đệ quy Python thuần)

    def f(x):
        if x <= 0:
            return 0
        if x == 1:
            return pts[1]
        if x not in memo:
            memo[x] = max(f(x - 1), f(x - 2) + pts[x])
        return memo[x]

    return f(len(pts) - 1)

for nums, expected in examples:
    print(nums, "->", earn_memo(nums), "(kỳ vọng", expected, ")")

[3, 4, 2] -> 6 (kỳ vọng 6 )
[2, 2, 3, 3, 3, 4] -> 9 (kỳ vọng 9 )


### Code nộp LeetCode (Cách 2)

```python
import sys

class Solution:
    def deleteAndEarn(self, nums: List[int]) -> int:
        sys.setrecursionlimit(50000)
        pts = [0] * (max(nums) + 1)
        for x in nums:
            pts[x] += x
        memo = {}

        def f(x):
            if x <= 0:
                return 0
            if x == 1:
                return pts[1]
            if x not in memo:
                memo[x] = max(f(x - 1), f(x - 2) + pts[x])
            return memo[x]

        return f(len(pts) - 1)
```

### Độ phức tạp

- Time: `O(n + M)` với `M = max(nums)`.
- Space: `O(M)` cho bộ nhớ đệm và ngăn xếp đệ quy.

### Cách 3 — DP mảng `dp[x]`

**Ý tưởng:** làm cùng công thức nhưng **từ dưới lên**, không đệ quy: điền `dp[0], dp[1], dp[2], ...` tuần tự. Không có đệ quy nên không lo giới hạn.

### Code demo

In [12]:
def earn_dp_array(nums):
    M = max(nums)
    pts = [0] * (M + 1)
    for x in nums:
        pts[x] += x
    dp = [0] * (M + 1)
    dp[1] = pts[1]
    for x in range(2, M + 1):
        dp[x] = max(dp[x - 1], dp[x - 2] + pts[x])
    return dp[M]

for nums, expected in examples:
    print(nums, "->", earn_dp_array(nums), "(kỳ vọng", expected, ")")

[3, 4, 2] -> 6 (kỳ vọng 6 )
[2, 2, 3, 3, 3, 4] -> 9 (kỳ vọng 9 )


### Code nộp LeetCode (Cách 3)

```python
class Solution:
    def deleteAndEarn(self, nums: List[int]) -> int:
        M = max(nums)
        pts = [0] * (M + 1)
        for x in nums:
            pts[x] += x
        dp = [0] * (M + 1)
        dp[1] = pts[1]
        for x in range(2, M + 1):
            dp[x] = max(dp[x - 1], dp[x - 2] + pts[x])
        return dp[M]
```

`M >= 1` nhờ ràng buộc `nums[i] >= 1`, nên `dp[1]` luôn hợp lệ.

### Độ phức tạp

- Time: `O(n + M)`.
- Space: `O(M)` cho `pts` và `dp` (có thể giảm `dp` còn hai biến).

### Cách 4 — DP cuộn trên các giá trị phân biệt (có xử lý khoảng trống)

**Ý tưởng khác biệt:** không cần mảng cỡ `M`. Duyệt các giá trị phân biệt theo thứ tự tăng dần. Giữ hai số: `take` = điểm tốt nhất **nếu lấy giá trị hiện tại**, `skip` = điểm tốt nhất **nếu không lấy**. Với giá trị mới `v` có `gain = v * cnt[v]`:

- Nếu `v == prev + 1` (liền kề): lấy `v` thì phải **không lấy** `prev`, nên `new_take = skip + gain`; không lấy `v`: `new_skip = max(take, skip)`.
- Nếu `v > prev + 1` (có khoảng trống): `v` **độc lập** với `prev` nên `new_take = max(take, skip) + gain`, `new_skip = max(take, skip)`.

Kết quả `max(take, skip)`. Cách này chạy tốt cả khi các giá trị rất lớn và thưa (ví dụ tới `10^9`), nơi mảng cỡ `M` không dùng được.

### Code demo

In [13]:
def earn_rolling(nums):
    cnt = Counter(nums)
    take = skip = 0
    prev = None
    for v in sorted(cnt):
        gain = v * cnt[v]
        if prev is not None and v == prev + 1:
            take, skip = skip + gain, max(take, skip)            # liền kề: phải bỏ prev
        else:
            take, skip = max(take, skip) + gain, max(take, skip)  # có khoảng trống: độc lập
        prev = v
    return max(take, skip)

for nums, expected in examples:
    print(nums, "->", earn_rolling(nums), "(kỳ vọng", expected, ")")
print("giá trị thưa:", earn_rolling([10 ** 9, 10 ** 9, 5, 7, 6]))

[3, 4, 2] -> 6 (kỳ vọng 6 )
[2, 2, 3, 3, 3, 4] -> 9 (kỳ vọng 9 )
giá trị thưa: 2000000012


### Code nộp LeetCode (Cách 4)

```python
from collections import Counter

class Solution:
    def deleteAndEarn(self, nums: List[int]) -> int:
        cnt = Counter(nums)
        take = skip = 0
        prev = None
        for v in sorted(cnt):
            gain = v * cnt[v]
            if prev is not None and v == prev + 1:
                take, skip = skip + gain, max(take, skip)
            else:
                take, skip = max(take, skip) + gain, max(take, skip)
            prev = v
        return max(take, skip)
```

### Độ phức tạp

- Time: `O(n + k log k)` với `k` là số giá trị phân biệt (`k log k` do sắp xếp).
- Space: `O(k)` cho `Counter`; phần DP chỉ dùng `O(1)`.

## 8. TRACE từng bước

Hàm `trace_delete_earn(nums)` in `pts`, rồi từng bước `dp[x]` kèm hai ứng viên (bỏ / lấy).

Bảng cho **ví dụ 2** `[2,2,3,3,3,4]` (`pts = [0, 0, 4, 9, 4]`):

| `x` | `pts[x]` | Bỏ: `dp[x-1]` | Lấy: `dp[x-2] + pts[x]` | `dp[x]` |
|---|---|---|---|---|
| 0 | 0 | - | - | 0 |
| 1 | 0 | - | - | 0 |
| 2 | 4 | `dp[1] = 0` | `dp[0] + 4 = 4` | 4 |
| 3 | 9 | `dp[2] = 4` | `dp[1] + 9 = 9` | 9 |
| 4 | 4 | `dp[3] = 9` | `dp[2] + 4 = 8` | 9 |

Đáp án `dp[4] = 9`.

In [14]:
def trace_delete_earn(nums):
    cnt = Counter(nums)
    print("đếm:", dict(sorted(cnt.items())))
    pts = build_points(nums)
    print("pts (chỉ số = giá trị):", pts)
    dp = [0] * len(pts)
    dp[1] = pts[1]
    print(f"dp[0] = 0, dp[1] = pts[1] = {dp[1]}")
    for x in range(2, len(pts)):
        skip, take = dp[x - 1], dp[x - 2] + pts[x]
        dp[x] = max(skip, take)
        choice = "LẤY" if take > skip else "BỎ"
        print(f"x={x}: bỏ = dp[{x - 1}] = {skip} | lấy = dp[{x - 2}] + pts[{x}] = {dp[x - 2]} + {pts[x]} = {take} -> {choice}, dp[{x}] = {dp[x]}")
    print("dp =", dp, "-> đáp án", dp[-1])
    return dp[-1]

trace_delete_earn([2, 2, 3, 3, 3, 4])

đếm: {2: 2, 3: 3, 4: 1}
pts (chỉ số = giá trị): [0, 0, 4, 9, 4]
dp[0] = 0, dp[1] = pts[1] = 0
x=2: bỏ = dp[1] = 0 | lấy = dp[0] + pts[2] = 0 + 4 = 4 -> LẤY, dp[2] = 4
x=3: bỏ = dp[2] = 4 | lấy = dp[1] + pts[3] = 0 + 9 = 9 -> LẤY, dp[3] = 9
x=4: bỏ = dp[3] = 9 | lấy = dp[2] + pts[4] = 4 + 4 = 8 -> BỎ, dp[4] = 9
dp = [0, 0, 4, 9, 9] -> đáp án 9


9

In [15]:
trace_delete_earn([3, 4, 2])
print()
trace_delete_earn([1, 1, 1, 3, 4])     # có khoảng trống giữa 1 và 3

đếm: {2: 1, 3: 1, 4: 1}
pts (chỉ số = giá trị): [0, 0, 2, 3, 4]
dp[0] = 0, dp[1] = pts[1] = 0
x=2: bỏ = dp[1] = 0 | lấy = dp[0] + pts[2] = 0 + 2 = 2 -> LẤY, dp[2] = 2
x=3: bỏ = dp[2] = 2 | lấy = dp[1] + pts[3] = 0 + 3 = 3 -> LẤY, dp[3] = 3
x=4: bỏ = dp[3] = 3 | lấy = dp[2] + pts[4] = 2 + 4 = 6 -> LẤY, dp[4] = 6
dp = [0, 0, 2, 3, 6] -> đáp án 6

đếm: {1: 3, 3: 1, 4: 1}
pts (chỉ số = giá trị): [0, 3, 0, 3, 4]
dp[0] = 0, dp[1] = pts[1] = 3
x=2: bỏ = dp[1] = 3 | lấy = dp[0] + pts[2] = 0 + 0 = 0 -> BỎ, dp[2] = 3
x=3: bỏ = dp[2] = 3 | lấy = dp[1] + pts[3] = 3 + 3 = 6 -> LẤY, dp[3] = 6
x=4: bỏ = dp[3] = 6 | lấy = dp[2] + pts[4] = 3 + 4 = 7 -> LẤY, dp[4] = 7
dp = [0, 3, 3, 6, 7] -> đáp án 7


7

Ở trace thứ ba (`[1,1,1,3,4]`): `pts = [0, 3, 0, 3, 4]`. Giá trị `2` vắng mặt nên `pts[2] = 0`; ở `x = 3`, ứng viên "lấy" là `dp[1] + 3 = 6`: tức lấy `1` (3 điểm) **và** `3` (3 điểm) cùng lúc, vì chúng cách nhau `2` nên không xung đột. Sau đó ở `x = 4` ứng viên "lấy" là `dp[2] + 4 = 7` (lấy `1` và `4`) thắng `dp[3] = 6`. Đáp án `7`.

## 9. Code nộp LeetCode

Ta nộp Cách 3 (DP mảng): rõ ràng nhất, đúng là "House Robber trên trục giá trị".

In [16]:
class Solution:
    def deleteAndEarn(self, nums: List[int]) -> int:
        M = max(nums)
        pts = [0] * (M + 1)
        for x in nums:
            pts[x] += x
        dp = [0] * (M + 1)
        dp[1] = pts[1]
        for x in range(2, M + 1):
            dp[x] = max(dp[x - 1], dp[x - 2] + pts[x])
        return dp[M]


for nums, expected in examples:
    got = Solution().deleteAndEarn(nums)
    print(nums, "->", got, "| kỳ vọng:", expected, "| OK" if got == expected else "| SAI")

[3, 4, 2] -> 6 | kỳ vọng: 6 | OK
[2, 2, 3, 3, 3, 4] -> 9 | kỳ vọng: 9 | OK


## 10. Vì sao đúng

**(a) Lấy giá trị nào thì lấy hết bản sao.** Xét một dãy thao tác tối ưu. Nếu ở một thời điểm ta xóa một bản sao của `v` thì mọi `v - 1`, `v + 1` bị xóa; các bản sao còn lại của `v` vẫn còn và xóa chúng không kéo thêm hậu quả nào (hàng xóm đã biến mất). Bỏ lại chúng chỉ làm mất điểm. Vì mọi điểm đều dương, ta lấy hết `cnt[v]` bản sao: điểm `v * cnt[v]`.

**(b) Tập giá trị được lấy.** Gọi `S` là tập các giá trị mà dãy thao tác có lấy. Nếu `v` và `v + 1` cùng thuộc `S`: lần đầu lấy một trong hai sẽ xóa sạch cái kia, vô lý. Vậy `S` **không có hai giá trị liền kề**. Ngược lại, mọi tập `S` không có hai giá trị liền kề đều thực hiện được: lấy lần lượt các bản sao của từng giá trị trong `S`; việc lấy `v` chỉ xóa `v ± 1 ∉ S`, nên không ảnh hưởng tới các giá trị khác của `S`.

**(c) Quy dẫn.** Vậy đáp án = `max` của `Σ_{v ∈ S} v * cnt[v]` qua mọi `S` không có hai giá trị liền kề. Đây là House Robber trên dãy `pts[1], pts[2], ..., pts[M]`.

**(d) Công thức DP.** `dp[x]` là tối ưu khi chỉ xét `1..x`. Giá trị `x` hoặc không thuộc `S` (đáp án `dp[x - 1]`), hoặc thuộc `S` (khi đó `x - 1 ∉ S`; phần còn lại là một tập hợp lệ trên `1..x - 2`, tối ưu `dp[x - 2]`, cộng `pts[x]`).

**(e) Giá trị vắng mặt.** `pts[x] = 0`; lấy hay không lấy cho cùng điểm, nên công thức vẫn đúng, và khoảng trống cho phép lấy hai giá trị hai bên (vì `dp[x - 2]` đã gồm giá trị `x - 2`).

Để kiểm chứng độc lập ta viết một **mô phỏng đúng luật trò chơi** (không dùng suy luận trên): thử mọi lựa chọn phần tử, xóa đúng một bản sao và mọi `v ± 1`, đệ quy có nhớ trên trạng thái (đa tập còn lại). Cách này chậm (`n <= 8`) nhưng không dựa vào lập luận (a)-(e).

In [17]:
def game_oracle(nums):
    @lru_cache(None)
    def play(state):
        if not state:
            return 0
        best = 0
        for v in set(state):
            rest = list(state)
            rest.remove(v)                                           # xóa đúng MỘT bản sao của v
            rest = tuple(y for y in rest if y != v - 1 and y != v + 1)   # xóa mọi v - 1 và v + 1
            best = max(best, v + play(rest))
        return best
    return play(tuple(sorted(nums)))

for nums, expected in examples:
    print(nums, "-> mô phỏng luật chơi:", game_oracle(nums), "(kỳ vọng", expected, ")")

rng = random.Random(740)
cnt_checked = 0
for _ in range(400):
    n = rng.randint(1, 8)
    nums = [rng.randint(1, rng.choice([3, 5, 8])) for _ in range(n)]
    assert Solution().deleteAndEarn(nums) == game_oracle(nums), nums
    cnt_checked += 1
print("lời giải khớp mô phỏng luật chơi trên", cnt_checked, "mảng ngẫu nhiên")

[3, 4, 2] -> mô phỏng luật chơi: 6 (kỳ vọng 6 )
[2, 2, 3, 3, 3, 4] -> mô phỏng luật chơi: 9 (kỳ vọng 9 )
lời giải khớp mô phỏng luật chơi trên 400 mảng ngẫu nhiên


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1 Quay lui | `O(φ^k)` (mũ theo số giá trị phân biệt `k`) | `O(k)` |
| 2 Đệ quy có nhớ | `O(n + M)` | `O(M)` |
| 3 DP mảng | `O(n + M)` | `O(M)` |
| 4 DP cuộn trên giá trị phân biệt | `O(n + k log k)` | `O(k)` |

Với `n <= 2 · 10^4` và `M <= 10^4` đều rất nhỏ. Ta đo thời gian ở kích thước tối đa.

In [18]:
rng = random.Random(2)
cases = {
    "ngẫu nhiên n=2*10^4, v<=10^4": [rng.randint(1, 10 ** 4) for _ in range(20000)],
    "toàn bằng nhau": [10 ** 4] * 20000,
    "mọi giá trị 1..10^4 (mỗi giá trị 2 lần)": [v for v in range(1, 10 ** 4 + 1) for _ in range(2)],
}
for name, nums in cases.items():
    res = []
    for fn in (earn_memo, earn_dp_array, earn_rolling):
        t0 = time.perf_counter()
        r = fn(nums)
        res.append((r, round(time.perf_counter() - t0, 4)))
    assert res[0][0] == res[1][0] == res[2][0]
    print(f"{name}: đáp án {res[0][0]} | memo {res[0][1]}s, mảng {res[1][1]}s, cuộn {res[2][1]}s")

ngẫu nhiên n=2*10^4, v<=10^4: đáp án 61028784 | memo 0.0082s, mảng 0.0036s, cuộn 0.0037s


toàn bằng nhau: đáp án 200000000 | memo 0.0071s, mảng 0.0023s, cuộn 0.0011s


mọi giá trị 1..10^4 (mỗi giá trị 2 lần): đáp án 50010000 | memo 0.0066s, mảng 0.0031s, cuộn 0.0036s


## 12. Bộ test

Các nhóm test: ví dụ của đề, edge case (một phần tử, toàn bằng nhau, hai giá trị liền kề, khoảng trống lớn), phản ví dụ (các trường hợp làm hỏng cách sai), và đối chiếu **nhiều cách** (kể cả mô phỏng luật chơi) trên input ngẫu nhiên có seed.

In [19]:
# 1) Ví dụ của đề
for nums, expected in examples:
    assert Solution().deleteAndEarn(nums) == expected

# 2) Edge case
edge = [
    ([1], 1),
    ([10000], 10000),
    ([5, 5, 5], 15),                       # toàn bằng nhau: lấy hết
    ([1, 2], 2),                           # liền kề: chỉ lấy một
    ([1, 3], 4),                           # khoảng trống: lấy cả hai
    ([2, 2, 3], 4),                        # lấy 2,2 (4) hơn lấy 3 (3)
    ([1, 1, 1, 3, 4], 7),
    ([1, 2, 3, 4, 5], 9),                  # 1 + 3 + 5
    ([2, 4, 6], 12),
    ([8, 10, 4, 9, 1, 3, 5, 9, 4, 10], 37),
]
for nums, expected in edge:
    assert Solution().deleteAndEarn(nums) == expected, (nums, expected)
print("ví dụ + edge OK")

ví dụ + edge OK


In [20]:
# 3) Đối chiếu 5 cách + mô phỏng luật chơi trên input nhỏ
rng = random.Random(740)
total = 0
for _ in range(1500):
    n = rng.randint(1, 8)
    top = rng.choice([2, 3, 5, 8, 12])
    nums = [rng.randint(1, top) for _ in range(n)]
    e = game_oracle(nums)
    assert earn_backtrack(nums)[0] == earn_memo(nums) == earn_dp_array(nums) == earn_rolling(nums) == Solution().deleteAndEarn(nums) == e, nums
    total += 1
print("1500 input nhỏ x 5 cách + mô phỏng luật chơi: OK")

# 4) Input vừa: đối chiếu các cách (bỏ mô phỏng vì quá chậm), quay lui chỉ với <= 22 giá trị phân biệt
for _ in range(300):
    n = rng.randint(1, 200)
    top = rng.choice([5, 15, 22, 100, 1000])
    nums = [rng.randint(1, top) for _ in range(n)]
    a = Solution().deleteAndEarn(nums)
    assert a == earn_memo(nums) == earn_rolling(nums)
    if len(set(nums)) <= 22:
        assert a == earn_backtrack(nums)[0]
print("input vừa OK")

# 5) Giới hạn tối đa
rng = random.Random(20000)
nums = [rng.randint(1, 10 ** 4) for _ in range(20000)]
assert Solution().deleteAndEarn(nums) == earn_memo(nums) == earn_rolling(nums)
assert Solution().deleteAndEarn([10 ** 4] * 20000) == 10 ** 4 * 20000
print("n = 2*10^4 OK")

# 6) Tính chất: đáp án <= tổng các phần tử và >= điểm lớn nhất của một giá trị đơn lẻ; đầu vào không bị sửa
rng = random.Random(6)
for _ in range(300):
    nums = [rng.randint(1, 30) for _ in range(rng.randint(1, 60))]
    copy = nums[:]
    r = Solution().deleteAndEarn(nums)
    assert nums == copy
    assert max(v * c for v, c in Counter(nums).items()) <= r <= sum(nums)
print("tính chất OK")

1500 input nhỏ x 5 cách + mô phỏng luật chơi: OK


input vừa OK
n = 2*10^4 OK
tính chất OK


## 13. Các lỗi người mới hay gặp

Ta cài vài hàm sai cố ý rồi tìm input nhỏ nhất làm chúng sai bằng `find_mismatch` (so với `game_oracle`).

In [21]:
def find_mismatch(fn, trials=3000, seed=21):
    rng = random.Random(seed)
    best = None
    for _ in range(trials):
        n = rng.randint(1, 7)
        nums = [rng.randint(1, rng.choice([4, 6])) for _ in range(n)]
        if fn(nums) != game_oracle(nums):
            if best is None or len(nums) < len(best):
                best = sorted(nums)
    return best

### Lỗi 1: chạy House Robber trên mảng `nums` thô

Sắp xếp `nums` rồi áp công thức "không lấy hai phần tử liền kề **trong mảng**" là sai: ràng buộc là về **giá trị liền kề**, không phải **vị trí liền kề**; các bản sao bằng nhau nằm cạnh nhau nhưng được phép lấy cùng nhau.

In [22]:
def wrong_raw_robber(nums):
    a = sorted(nums)
    prev2 = prev1 = 0
    for x in a:                                  # SAI: coi mỗi phần tử là một "nhà"
        prev2, prev1 = prev1, max(prev1, prev2 + x)
    return prev1

print("[6,6] sai:", wrong_raw_robber([6, 6]), "| đúng:", Solution().deleteAndEarn([6, 6]), "(hai số 6 nằm cạnh nhau nhưng được lấy cùng nhau)")
print("[2,2,3,3,3,4] sai:", wrong_raw_robber([2, 2, 3, 3, 3, 4]), "| đúng:", Solution().deleteAndEarn([2, 2, 3, 3, 3, 4]), "(trùng hợp, ví dụ này chưa lộ lỗi)")
print("input nhỏ nhất làm sai:", find_mismatch(wrong_raw_robber))

[6,6] sai: 6 | đúng: 12 (hai số 6 nằm cạnh nhau nhưng được lấy cùng nhau)
[2,2,3,3,3,4] sai: 9 | đúng: 9 (trùng hợp, ví dụ này chưa lộ lỗi)
input nhỏ nhất làm sai: [6, 6]


**Vì sao sai:** các bản sao của cùng một giá trị phải được **gom lại** thành một "nhà" có điểm `v * cnt[v]`; ràng buộc xung đột chỉ áp lên các **giá trị** liền kề. **Code đúng:** xây `pts[v] += v` rồi mới chạy DP.

### Lỗi 2: quên nhân với số lần xuất hiện

Dùng `pts[v] = v` (mỗi giá trị chỉ góp một lần) thay vì `v * cnt[v]`.

In [23]:
def wrong_no_count(nums):
    M = max(nums)
    pts = [0] * (M + 1)
    for x in set(nums):                          # SAI: set bỏ mất số lần xuất hiện
        pts[x] = x
    dp = [0] * (M + 1)
    dp[1] = pts[1]
    for x in range(2, M + 1):
        dp[x] = max(dp[x - 1], dp[x - 2] + pts[x])
    return dp[M]

print("[2,2,3,3,3,4] sai:", wrong_no_count([2, 2, 3, 3, 3, 4]), "| đúng:", Solution().deleteAndEarn([2, 2, 3, 3, 3, 4]))
print("input nhỏ nhất làm sai:", find_mismatch(wrong_no_count))

[2,2,3,3,3,4] sai: 6 | đúng: 9
input nhỏ nhất làm sai: [6, 6]


**Vì sao sai:** lấy giá trị `v` là lấy được **mọi** bản sao nên điểm là `v * cnt[v]`. **Code đúng:** `pts[x] += x` cho từng phần tử (cộng dồn).

### Lỗi 3: coi mọi giá trị phân biệt kề nhau trong danh sách đã sắp xếp là xung đột

Khi chạy DP trên **các giá trị phân biệt đã sắp xếp** (như Cách 4) mà quên kiểm tra `v == prev + 1`, hai giá trị cách xa nhau (ví dụ `1` và `3`) bị coi là xung đột.

In [24]:
def wrong_no_gap(nums):
    cnt = Counter(nums)
    prev2 = prev1 = 0
    for v in sorted(cnt):                        # SAI: mọi giá trị liền kề trong danh sách đều bị coi là xung đột
        prev2, prev1 = prev1, max(prev1, prev2 + v * cnt[v])
    return prev1

print("[1,3] sai:", wrong_no_gap([1, 3]), "| đúng:", Solution().deleteAndEarn([1, 3]))
print("input nhỏ nhất làm sai:", find_mismatch(wrong_no_gap))

[1,3] sai: 3 | đúng: 4


input nhỏ nhất làm sai: [4, 6]


**Vì sao sai:** `1` và `3` không xung đột (lấy `1` chỉ xóa `0` và `2`), nên đáp án là `1 + 3 = 4`, không phải `3`. **Code đúng:** chỉ áp ràng buộc khi `v == prev + 1` (Cách 4), hoặc dùng mảng theo giá trị (Cách 3) để các khoảng trống tự có `pts = 0`.

### Lỗi 4: tham lam "lấy giá trị nhiều điểm nhất trước"

Có vẻ hợp lý: luôn lấy giá trị có `v * cnt[v]` lớn nhất, xóa hàng xóm, lặp lại. Nhưng lấy cái lớn nhất có thể **phá** hai giá trị hai bên cộng lại còn lớn hơn.

In [25]:
def wrong_greedy(nums):
    cnt = Counter(nums)
    total = 0
    while cnt:
        v = max(cnt, key=lambda x: (x * cnt[x], x))      # giá trị nhiều điểm nhất
        total += v * cnt[v]
        for u in (v - 1, v, v + 1):
            cnt.pop(u, None)
    return total

bad = find_mismatch(wrong_greedy)
print("input nhỏ nhất làm sai:", bad)
print("tham lam:", wrong_greedy(bad), "| đúng:", Solution().deleteAndEarn(bad))
ex = [2] * 4 + [3] * 3 + [4] * 2                      # điểm: 2 -> 8, 3 -> 9, 4 -> 8
print("ví dụ có chủ đích: điểm", {2: 8, 3: 9, 4: 8}, "| tham lam:", wrong_greedy(ex), "| đúng:", Solution().deleteAndEarn(ex))

input nhỏ nhất làm sai: [1, 1, 2, 2, 3]
tham lam: 4 | đúng: 5
ví dụ có chủ đích: điểm {2: 8, 3: 9, 4: 8} | tham lam: 9 | đúng: 16


**Vì sao sai:** ở ví dụ có chủ đích, điểm của `2`, `3`, `4` lần lượt là `8, 9, 8`: tham lam lấy `3` (9 điểm) và xóa luôn `2` và `4`, còn đáp án đúng là lấy `2` và `4` (`8 + 8 = 16`). **Code đúng:** quy hoạch động (cân nhắc cả hai lựa chọn "lấy" và "bỏ" ở mỗi giá trị) thay vì chọn tham lam cục bộ.

## 14. Mẹo & nhận dạng pattern

- Thấy "chọn một phần tử thì **cấm** các phần tử liền kề (theo vị trí hoặc theo giá trị)" → **House Robber**.
- Khi ràng buộc nằm trên **giá trị** (`v ± 1`), hãy **gom** các phần tử cùng giá trị trước (đếm), rồi thay thế bài toán bằng một dãy theo **trục giá trị**.
- Nếu giá trị nhỏ (`<= 10^4`) dùng **mảng chỉ mục theo giá trị**: khoảng trống tự có điểm `0`. Nếu giá trị lớn và thưa, dùng **danh sách giá trị phân biệt đã sắp xếp** và kiểm tra `v == prev + 1`.
- Công thức quen thuộc: `dp[x] = max(dp[x - 1], dp[x - 2] + pts[x])`; có thể nén còn hai biến.
- Đệ quy sâu `10^4` cần nâng `sys.setrecursionlimit` (và nên tự cài `memo` bằng `dict` thay vì `lru_cache`); DP bottom-up thì không lo.
- Nhớ đối chiếu một **mô phỏng đúng luật** cho input rất nhỏ để bắt lỗi quy dẫn.
- Nghi ngờ tham lam thì thử phản ví dụ điểm `{2: 8, 3: 9, 4: 8}`: tham lam "nhiều điểm nhất trước" thất bại.

## 15. Biến thể

### 15.1. House Robber gốc (LeetCode 198)

Chính hàm `house_robber` ở mục 3.3: `prev2, prev1 = prev1, max(prev1, prev2 + x)`.

### 15.2. House Robber II — các nhà xếp **vòng tròn** (LeetCode 213)

Nhà đầu và nhà cuối liền kề nên không lấy cả hai: lấy `max` của hai trường hợp "bỏ nhà đầu" và "bỏ nhà cuối".

In [26]:
def house_robber_circle(a):
    if len(a) == 1:
        return a[0]
    return max(house_robber(a[1:]), house_robber(a[:-1]))

def house_robber_circle_brute(a):
    n, best = len(a), 0
    for mask in range(1 << n):
        ok = all(not (mask >> i & 1 and mask >> ((i + 1) % n) & 1) for i in range(n)) if n > 1 else True
        if ok:
            best = max(best, sum(a[i] for i in range(n) if mask >> i & 1))
    return best

rng = random.Random(152)
for _ in range(300):
    a = [rng.randint(0, 9) for _ in range(rng.randint(1, 10))]
    assert house_robber_circle(a) == house_robber_circle_brute(a), a
print(house_robber_circle([2, 3, 2]), house_robber_circle([1, 2, 3, 1]))     # 3, 4

3 4


### 15.3. Tổng quát hóa: lấy `v` thì cấm mọi giá trị trong khoảng `(v - k, v + k)`

Khi `k = 2` ta có đề gốc (cấm `v ± 1`). Với `k` tổng quát: sau khi lấy `v`, cấm `v - k + 1, ..., v + k - 1`. Công thức DP trở thành `dp[x] = max(dp[x - 1], dp[max(x - k, 0)] + pts[x])` (lấy `x` thì nhảy về `dp[x - k]`).

In [27]:
def earn_gap(nums, k):
    M = max(nums)
    pts = [0] * (M + 1)
    for x in nums:
        pts[x] += x
    dp = [0] * (M + 1)
    for x in range(1, M + 1):
        dp[x] = max(dp[x - 1], dp[max(x - k, 0)] + pts[x])
    return dp[M]

def earn_gap_brute(nums, k):
    cnt = Counter(nums)
    vals = sorted(cnt)
    best = 0
    for mask in range(1 << len(vals)):
        chosen = [vals[i] for i in range(len(vals)) if mask >> i & 1]
        if all(b - a >= k for a, b in zip(chosen, chosen[1:])):     # hai giá trị được chọn cách nhau >= k
            best = max(best, sum(v * cnt[v] for v in chosen))
    return best

rng = random.Random(153)
for _ in range(400):
    nums = [rng.randint(1, 12) for _ in range(rng.randint(1, 15))]
    k = rng.randint(1, 4)
    assert earn_gap(nums, k) == earn_gap_brute(nums, k), (nums, k)
    if k == 2:
        assert earn_gap(nums, 2) == Solution().deleteAndEarn(nums)
print("k = 1 (không cấm gì):", earn_gap([1, 2, 3], 1), "| k = 2 (đề gốc):", earn_gap([1, 2, 3], 2), "| k = 3:", earn_gap([1, 2, 3], 3))

k = 1 (không cấm gì): 6 | k = 2 (đề gốc): 4 | k = 3: 3


### 15.4. Bài liên quan

- LeetCode 198 House Robber, 213 House Robber II, 337 House Robber III (trên cây).
- LeetCode 746 Min Cost Climbing Stairs, 70 Climbing Stairs (cùng họ DP một chiều).
- LeetCode 2140 Solving Questions With Brainpower (chọn một câu thì bỏ qua `k` câu sau).
- LeetCode 1388 Pizza With 3n Slices (chọn không liền kề có giới hạn số lượng).

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1 Quay lui | chọn/bỏ từng giá trị phân biệt | `O(φ^k)` | `O(k)` | chỉ để hiểu đề / oracle nhỏ |
| 2 Đệ quy có nhớ | `f(x) = max(f(x-1), f(x-2) + pts[x])` | `O(n + M)` | `O(M)` | khi muốn viết top-down |
| 3 DP mảng | điền `dp[0..M]` | `O(n + M)` | `O(M)` | **cách nộp**, rõ ràng nhất |
| 4 DP cuộn trên giá trị phân biệt | `take/skip`, kiểm tra `v == prev + 1` | `O(n + k log k)` | `O(k)` | giá trị lớn và thưa |

## 17. Bài tập vận dụng

Làm lần lượt từ dễ đến khó. Gợi ý ở mục 18, đáp án ở mục 19.

### Bài 1 (đoán output)

Dùng `build_points` (mục 3.2). Đoán `build_points([1, 1, 2, 5, 5])` (danh sách 6 số), điền `ex1_answer`. **Không** chạy trước khi đoán.

In [28]:
ex1_nums = [1, 1, 2, 5, 5]
ex1_answer = None       # ví dụ [0, 0, 0, 0, 0, 0]  <- chỉ là mẫu định dạng

### Bài 2 (tính tay)

Tính tay đáp án của `deleteAndEarn` cho bốn mảng: `[1, 2, 3, 4, 5]`, `[1, 1, 1, 3, 4]`, `[5, 5, 5]`, `[2, 4, 6]`. Điền `ex2_answer` (danh sách 4 số theo thứ tự đó).

In [29]:
ex2_arrays = [[1, 2, 3, 4, 5], [1, 1, 1, 3, 4], [5, 5, 5], [2, 4, 6]]
ex2_answer = None       # ví dụ [0, 0, 0, 0]

### Bài 3 (điền TODO)

Hoàn thành `ex3_earn`: hai chỗ `TODO` là bước gom điểm và công thức DP. Cell này **không gọi** hàm.

In [30]:
def ex3_earn(nums):
    M = max(nums)
    pts = [0] * (M + 1)
    for x in nums:
        pass                         # TODO 1: gom điểm cho giá trị x
    dp = [0] * (M + 1)
    dp[1] = pts[1]
    for x in range(2, M + 1):
        dp[x] = 0                    # TODO 2: max(bỏ x, lấy x)
    return dp[M]

### Bài 4 (tự viết lại)

Tự viết **không nhìn** hàm `ex4_rewrite(nums)` bằng **Cách 4** (duyệt các giá trị phân biệt đã sắp xếp với hai biến `take`, `skip`). Để `pass` nếu chưa làm.

In [31]:
def ex4_rewrite(nums):
    pass

### Bài 5 (biến thể / mini challenge)

Viết `ex5_chosen(nums)`: trả về **danh sách các giá trị** (tăng dần) được lấy trong một phương án tối ưu (không có hai giá trị liền kề, tổng `v * cnt[v]` bằng đáp án). Gợi ý: lưu mảng `dp` rồi truy ngược.

In [32]:
def ex5_chosen(nums):
    pass

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: mảng có độ dài `max + 1 = 6` và chỉ số là giá trị.
- Mức 2: mỗi phần tử `x` cộng `x` vào `pts[x]`.
- Mức 3: `pts[1] = 2`, `pts[2] = 2`, `pts[5] = 10`, còn lại `0`.

**Bài 2**
- Mức 1: tính điểm `pts` rồi chọn tập giá trị không liền kề.
- Mức 2: `[1,2,3,4,5]`: `1 + 3 + 5`; `[1,1,1,3,4]`: lấy `1` và `4`.
- Mức 3: đáp án `[9, 7, 15, 12]`.

**Bài 3**
- Mức 1: gom điểm bằng cộng dồn theo chỉ số giá trị.
- Mức 2: TODO 1: `pts[x] += x`; TODO 2: `max(dp[x - 1], dp[x - 2] + pts[x])`.
- Mức 3: giống hệt Cách 3.

**Bài 4**
- Mức 1: `Counter`, `sorted`, hai biến `take`, `skip`.
- Mức 2: liền kề thì `take = skip + gain`; còn khoảng trống thì `take = max(take, skip) + gain`.
- Mức 3: `skip` mới luôn là `max(take, skip)` cũ.

**Bài 5**
- Mức 1: tính `dp[x]` như Cách 3.
- Mức 2: truy ngược từ `x = M`: nếu `dp[x] == dp[x - 1]` thì bỏ `x` (lùi 1), ngược lại lấy `x` (lùi 2).
- Mức 3: chỉ ghi giá trị `x` có `cnt > 0` (với `x` vắng mặt, `pts = 0`, không cần ghi); trả về danh sách tăng dần.

## 19. Đáp án tham khảo

### Đáp án Bài 1

In [33]:
ex1_solution = [0, 2, 2, 0, 0, 10]
assert build_points(ex1_nums) == ex1_solution
print("Bài 1 OK:", ex1_solution)

Bài 1 OK: [0, 2, 2, 0, 0, 10]


### Đáp án Bài 2

In [34]:
ex2_solution = [9, 7, 15, 12]
assert [Solution().deleteAndEarn(a) for a in ex2_arrays] == ex2_solution
assert [game_oracle(a) for a in ex2_arrays] == ex2_solution
print("Bài 2 OK:", ex2_solution)

Bài 2 OK: [9, 7, 15, 12]


### Đáp án Bài 3

In [35]:
def ex3_solution(nums):
    M = max(nums)
    pts = [0] * (M + 1)
    for x in nums:
        pts[x] += x
    dp = [0] * (M + 1)
    dp[1] = pts[1]
    for x in range(2, M + 1):
        dp[x] = max(dp[x - 1], dp[x - 2] + pts[x])
    return dp[M]

rng = random.Random(3)
for _ in range(300):
    nums = [rng.randint(1, 12) for _ in range(rng.randint(1, 20))]
    assert ex3_solution(nums) == earn_rolling(nums)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [36]:
def ex4_solution(nums):
    cnt = Counter(nums)
    take = skip = 0
    prev = None
    for v in sorted(cnt):
        gain = v * cnt[v]
        if prev is not None and v == prev + 1:
            take, skip = skip + gain, max(take, skip)
        else:
            take, skip = max(take, skip) + gain, max(take, skip)
        prev = v
    return max(take, skip)

rng = random.Random(4)
for _ in range(300):
    nums = [rng.randint(1, 15) for _ in range(rng.randint(1, 20))]
    assert ex4_solution(nums) == Solution().deleteAndEarn(nums)
assert ex4_solution([10 ** 9, 10 ** 9, 5, 7, 6]) == earn_rolling([10 ** 9, 10 ** 9, 5, 7, 6])
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [37]:
def ex5_solution(nums):
    M = max(nums)
    cnt = Counter(nums)
    pts = [0] * (M + 1)
    for x in nums:
        pts[x] += x
    dp = [0] * (M + 1)
    dp[1] = pts[1]
    for x in range(2, M + 1):
        dp[x] = max(dp[x - 1], dp[x - 2] + pts[x])
    chosen, x = [], M
    while x >= 1:
        if x >= 2 and dp[x] == dp[x - 1]:
            x -= 1                                   # bỏ x
        else:
            if cnt[x] > 0:
                chosen.append(x)
            x -= 2                                   # lấy x: bỏ luôn x - 1
    return sorted(chosen)

rng = random.Random(5)
for _ in range(500):
    nums = [rng.randint(1, 14) for _ in range(rng.randint(1, 20))]
    chosen = ex5_solution(nums)
    cnt = Counter(nums)
    assert all(b - a >= 2 for a, b in zip(chosen, chosen[1:])), (nums, chosen)       # không liền kề
    assert sum(v * cnt[v] for v in chosen) == Solution().deleteAndEarn(nums), (nums, chosen)
print("Bài 5 OK, ví dụ:", ex5_solution([2, 2, 3, 3, 3, 4]), ex5_solution([1, 1, 1, 3, 4]))

Bài 5 OK, ví dụ: [3] [1, 4]


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
class Solution:
    def deleteAndEarn(self, nums: List[int]) -> int:
        M = max(nums)
        pts = [0] * (M + 1)
        for x in nums:
            pts[x] += x                              # gom: lấy giá trị x được x * (số lần x)
        dp = [0] * (M + 1)
        dp[1] = pts[1]
        for x in range(2, M + 1):
            dp[x] = max(dp[x - 1], dp[x - 2] + pts[x])   # bỏ x hoặc lấy x (bỏ x - 1)
        return dp[M]
```

Câu thần chú: **"gom theo giá trị, lấy v thì bỏ v - 1, House Robber trên trục giá trị."**

### Checklist trước khi submit

- [ ] Đã **gom** các bản sao: `pts[x] += x` (không phải `pts[x] = x`, không chạy DP trên `nums` thô).
- [ ] Ràng buộc xung đột chỉ áp cho **giá trị liền kề** (`v` và `v + 1`).
- [ ] Khoảng trống giữa các giá trị: `pts = 0` (mảng) hoặc kiểm tra `v == prev + 1` (danh sách phân biệt).
- [ ] Công thức `dp[x] = max(dp[x - 1], dp[x - 2] + pts[x])`; cơ sở `dp[0] = 0`, `dp[1] = pts[1]`.
- [ ] Mảng đủ chỉ số `0..max(nums)` (kích thước `max(nums) + 1`).
- [ ] Không dùng tham lam "lấy nhiều điểm nhất trước".
- [ ] Test: một phần tử, toàn bằng nhau, `[1,2]`, `[1,3]`, ví dụ 1-2 của đề.
- [ ] Skeleton đúng: `class Solution`, method `deleteAndEarn(self, nums: List[int]) -> int`.